# torch.compile 原理介绍


## 1. 简介


torch.compile()是PyTorch 2.0+推出的核心优化接口，通过“动态图捕获+静态图优化+高效代码生成”的方式显著加速模型训练和推理任务。Ascend Extension for PyTorch在2.6.0以上版本已支持该特性，为用户提供三种常用的backend配置选项，分别是torch.compile(backend="inductor")、torch.compile(backend="npugraphs")和torch.compile(backend="npugraph_ex")。

下图展示了 torch.compile 的整体架构，Dynamo 从用户代码中捕获 FX Graph，再交由后端编译器（如 Inductor、npugraph_ex）进行优化和代码生成：

**图 1**  torch.compile整体架构图

<img src="./images/pytorch.png" width="600"> <a id="fig1"></a>


## 2. torch.compile 底层原理


### 2.1 FX 图定义

torch FX 定义了一种图结构表达，并且能够进行python代码生成，是一个python to python的代码转换工具。

可以通过以下示例了解FX node的含义以及最终生成的python代码

In [ ]:
import torch
import torch.fx as fx

class MyModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.param = torch.nn.Parameter(torch.rand(3,4))
        self.linear = torch.nn.Linear(4, 5)

    def forward(self, x):
        return torch.topk(torch.sum(
            self.linear(x + self.linear.weight).relu(), dim=-1), 3)
        

m = MyModule()
gm = torch.fx.symbolic_trace(m)

print("=" * 60)
print("FX Graph 节点结构:")
print("=" * 60)
print(gm.graph)

print("=" * 60)
print("生成的 Python 代码:")
print("=" * 60)
print(gm.code)


上述代码运行后，分别打印了 FX Graph 的节点结构和由图生成的等价 Python 代码。输出内容如下：

```
============================================================
FX Graph 节点结构:
============================================================
graph():
    %x : ［num_users=1］ = placeholder［target=x］
    %linear_weight : ［num_users=1］ = get_attr［target=linear.weight］
    %add : ［num_users=1］ = call_function［target=operator.add］(args = (%x, %linear_weight), kwargs = {})
    %linear : ［num_users=1］ = call_module［target=linear］(args = (%add,), kwargs = {})
    %relu : ［num_users=1］ = call_method［target=relu］(args = (%linear,), kwargs = {})
    %sum_1 : ［num_users=1］ = call_function［target=torch.sum］(args = (%relu,), kwargs = {dim: -1})
    %topk : ［num_users=1］ = call_function［target=torch.topk］(args = (%sum_1, 3), kwargs = {})
    return topk

============================================================
生成的 Python 代码:
============================================================
def forward(self, x):
    linear_weight = self.linear.weight
    add = x + linear_weight;  x = linear_weight = None
    linear = self.linear(add);  add = None
    relu = linear.relu();  linear = None
    sum_1 = torch.sum(relu, dim = -1);  relu = None
    topk = torch.topk(sum_1, 3);  sum_1 = None
    return topk
```


下面对输出内容进行解读。

**FX Graph 节点结构**

输出中的图结构描述了计算流程中每一步的操作。以 `%add` 节点为例：

```python

%add : ［num_users=1］ = call_function［target=operator.add］(args = (%x, %linear_weight), kwargs = {})

```

各字段含义如下：

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">字段</th>
    <th align="left">说明</th>
    <th align="left">示例值</th>
  </tr>
  <tr>
    <td align="left"><strong>name</strong></td>
    <td align="left">节点名称，自动生成或由操作推断</td>
    <td align="left">`add`、`linear`、`relu`、`sum_1`、`topk`</td>
  </tr>
  <tr>
    <td align="left"><strong>num_users</strong></td>
    <td align="left">该节点的输出被多少个后续节点使用</td>
    <td align="left">`1` 表示只有 1 个下游节点依赖它</td>
  </tr>
  <tr>
    <td align="left"><strong>op</strong></td>
    <td align="left">操作类型（见下方说明）</td>
    <td align="left">`placeholder`、`get_attr`、`call_function`、`call_module`、`call_method`、`output`</td>
  </tr>
  <tr>
    <td align="left"><strong>target</strong></td>
    <td align="left">操作目标，函数引用 / 模块名 / 属性名</td>
    <td align="left">`operator.add`、`torch.sum`、`linear.weight`、`linear`</td>
  </tr>
  <tr>
    <td align="left"><strong>args</strong></td>
    <td align="left">位置参数，包含对其他节点的引用（即<strong>边</strong>）</td>
    <td align="left">`(%x, %linear_weight)` 表示 `add` 的输入来自 `x` 和 `linear_weight` 两个节点</td>
  </tr>
  <tr>
    <td align="left"><strong>kwargs</strong></td>
    <td align="left">关键字参数</td>
    <td align="left">`{dim: -1}` 表示 `torch.sum` 沿最后一个维度求和</td>
  </tr>
</table>


**op 操作类型说明**：

- **`placeholder`**：图的输入参数，如 `x`
- **`get_attr`**：从模块中获取属性（如权重 `linear.weight`）
- **`call_function`**：调用自由函数，如 `operator.add`、`torch.sum`、`torch.topk`
- **`call_module`**：调用子模块，如 `nn.Linear`，target 为模块名 `linear`
- **`call_method`**：调用张量方法，如 `.relu()`，target 为方法名 `relu`
- **`output`**：图的返回值

**等价 Python 代码**

输出的下半部分是通过 `gm.code` 打印的等价 Python 代码。可以看到，生成的代码与用户原始的 `forward` 方法功能完全等价，但所有操作都被展开为显式的、可分析和优化的步骤。后端编译器（如 Inductor）可以在此基础上进行算子融合、内存优化等变换。

### 2.2 Dynamo 介绍

TorchDynamo（简称 Dynamo）是 PyTorch 2.0 引入的一个 **Python 级别的即时编译器（JIT）**。它的设计初衷是在不修改用户代码的前提下，显著加速 PyTorch 程序的执行。不同于传统深度学习框架依赖的中间表示（IR）或静态图，Dynamo 选择更底层、更通用的切入方式——直接与 Python 解释器交互，将 Python 的动态灵活性与编译优化的高性能结合起来，是 `torch.compile` 背后的核心技术。

**核心原理：从字节码到机器码**

Dynamo 的核心能力可以概括为：通过重写 Python 字节码，将关键计算提取为图，并交由后端编译器生成高效的机器码。其实现高度依赖 CPython 的两个底层机制：

- **PEP 523（帧评估钩子）**：Dynamo 利用这个 API 在 CPython 执行每个函数栈帧（PyFrameObject）前设置一个钩子，从而获取控制权。这使得 Dynamo 能够在原始字节码运行之前对其进行拦截和分析，为后续的“即时编译”提供了切入点。

- **字节码分析与图捕获**：一旦钩子被触发，Dynamo 会启动一个内置的解释器来扫描当前函数的 Python 字节码。它会识别出与 torch 相关的张量操作，将这些操作序列提取并构建成一个 **FX Graph**。构建好的 FX Graph 随后交给可定制的后端编译器（如 Inductor）进行优化和编译，最终生成高效的机器码。

Dynamo 使得开发者可以方便地尝试不同的编译器后端来加速 PyTorch 代码，底层 API `torch._dynamo.optimize()` 由 `torch.compile()` 封装以方便使用。`TorchInductor` 是 Dynamo 支持的后端之一，它将 FX Graph 编译为 GPU 上的 [Triton](https://github.com/openai/triton) 代码或 CPU 上的 [C++/OpenMP](https://www.openmp.org/) 代码。

下图展示了常规 Python 执行流程与 TorchDynamo 介入后的执行流程：

**图 3**  TorchDynamo执行流程对比

<img src="./images/TorchDynamo.png" width="700"><a id="fig2"></a>


### 2.3 AOTAutograd

**AOTAutograd**（Ahead-Of-Time Autograd）是 torch.compile 编译流水线中的关键中间层，位于 Dynamo 与后端编译器（如 Inductor）之间。它的核心职责是将反向传播（backward）计算提前捕获并编译，从而实现前向与反向的联合优化。

**工作原理**：

1. **前向图捕获**：Dynamo 产出的 FX Graph 仅包含前向计算。AOTAutograd 利用 PyTorch 的 autograd 引擎，通过符号化追踪（symbolic tracing）生成对应的反向计算图。
2. **图分割**：将前向图和反向图拆分为多个独立的子图，交给后端编译器独立编译。
3. **联合优化**：后端编译器（如 Inductor）可以同时看到前向和反向计算，进行跨前后向的算子融合优化，减少内存访问和临时变量分配。

**为什么需要 AOTAutograd**：

- 传统的 Eager 模式中，反向传播在运行时动态构建，无法进行前后向联合优化
- AOTAutograd 将反向计算提前固定，使后端编译器能够对前后向进行整体优化，产生更高效的编译代码
- 这种“提前编译反向”的方式是 torch.compile 相对于传统 JIT 的一个重要优势


**前向图与反向图的差异**：

- **前向图（Forward Graph）**：描述用户代码的前向计算逻辑，操作顺序与用户代码一致（如 `linear → relu → sum`）。AOTAutograd 会将前向图进一步分解为底层 ATen 算子，并在前向图中保存反向传播所需的中间值（如 relu 的输出、转置后的权重等），避免反向时重新计算。

- **反向图（Backward Graph）**：AOTAutograd 通过 autograd 引擎自动生成，操作顺序与前向相反（如 `sum 的反向 → relu 的反向 → linear 的反向`）。反向图的输入包括前向图保存的中间值和上游传来的梯度（tangents），输出为各参数的梯度。



### 2.4 Backend

**Backend（后端）** 是 torch.compile 编译流水线的最后一环，负责将优化后的计算图转换为可执行的高效代码。不同的后端采用不同的编译策略，适合不同的场景。

Ascend Extension for PyTorch 提供了三种常用的 backend 配置选项：

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">后端</th>
    <th align="left">开启方式</th>
    <th align="left">核心机制</th>
    <th align="left">适用场景</th>
  </tr>
  <tr>
    <td align="left"><strong>Inductor</strong></td>
    <td align="left">`backend="inductor"`</td>
    <td align="left">Dynamo + Inductor 协同，算子融合与代码生成（Triton/MLIR/DVM）</td>
    <td align="left">迭代次数多、单步计算量中等的场景</td>
  </tr>
  <tr>
    <td align="left"><strong>NPUGraphs</strong></td>
    <td align="left">`backend="npugraphs"`</td>
    <td align="left">利用 NPUGraphs 技术，彻底消除 NPU 任务启动开销和 CPU-NPU 同步开销</td>
    <td align="left">host bound 且 kernel 调用频繁但输入形状固定的场景</td>
  </tr>
  <tr>
    <td align="left"><strong>NPUGraph_EX</strong></td>
    <td align="left">`backend="npugraph_ex"`</td>
    <td align="left">基于 ACLGraph 调度和 FX 图优化，大模型推理加速</td>
    <td align="left">大模型在线推理场景</td>
  </tr>
</table>


**Inductor 后端**：以降低 Python 开销和 kernel 启动开销为核心，通过 Dynamo + Inductor 协同，在不改变模型逻辑的前提下，自动进行算子融合和生成，提升训练或推理的吞吐量，尤其适合迭代次数多、单步计算量中等的场景。支持三种代码生成模式：

- **Triton 模式**（默认）：基于 Triton-Ascend 生成融合算子。关于 Triton-Ascend 的详细介绍，可以参考 [Triton-Ascend 官方仓库](https://gitcode.com/Ascend/triton-ascend)。
- **MLIR 模式**：通过 `torch.compile(backend="inductor", options={"npu_backend": "mlir"})` 使能，基于 Torch-MLIR 生成融合算子。关于 Torch-MLIR 的详细介绍，可以参考 [Torch-MLIR 官方仓库](https://github.com/llvm/torch-mlir)。
- **DVM 模式**：通过 `torch.compile(backend="inductor", options={"npu_backend": "dvm"})` 使能，基于 DVM 生成融合算子。关于 DVM 的详细介绍，可以参考 [DVM 官方仓库](https://gitcode.com/mindspore/dvm/tree/master)。

**NPUGraphs 后端**：利用 NPUGraphs 技术，彻底消除 NPU 任务的启动开销和 CPU 至 NPU 同步开销，适合 eager 模式存在 host bound 且 kernel 调用频繁但输入形状固定的场景，整体功能与 `backend="cudagraphs"` 一致。如需了解 NPUGraph 的工作原理、核心优势、API 详解和更多使用示例，请参阅 [NPUGraph](./01.03_npugraph_intro.ipynb)。

**NPUGraph_EX 后端**：基于 ACLGraph 调度和 FX 图优化，对大模型推理进行加速，并与主流服务化框架快速、无缝对接。

## 3. 使用指导

接口原型：

```python
def compile(model, *, fullgraph = False, dynamic = None, backend = "inductor", mode = None, options = None, disable = False)
```

参数说明：

- **model**: 必选参数，要编译的模型或者参数。
- **fullgraph**：可选参数，是否强制整图编译，默认值为False。
- **dynamic**：可选参数，是否需要动态shape编译，默认值为None。
- **backend**：可选参数，编译后端，支持inductor、npugraphs和npugraph_ex，默认值为inductor。
- **mode**：可选参数，编译模式，目前支持None（默认值）和"reduce-overhead"。
- **options**：可选参数，编译选项。

    - inductor和npugraphs支持以下参数：
      - triton.cudagraphs
      - trace.enabled
      - enable\_shape\_handling
      - npu\_backend
    - npugraph_ex支持的参数和详细使用指导请参考《PyTorch图模式使用(TorchAir)》中的[npugraph_ex后端](./01.04_npugraph_ex_basic_concepts.ipynb)。

- **disable**：可选参数，是否关闭torch.compile能力，默认值为False。

该接口详情可参考原生[torch.compile](https://docs.pytorch.org/docs/stable/generated/torch.compile.html)。


## 4. 约束说明


1. 优化器（optimizer）通常不入图，优化器的step()包含Python侧动态逻辑（如学习率调度、梯度累积、自适应更新规则），难以被静态图捕获。
2. torch.compile(backend="npugraphs")必须固定输入形状（捕获后无法修改 batch\_size、序列长度等）。torch.compile(backend="inductor")支持动态形状，但会触发重新编译（增加开销），建议尽量固定形状。
3. 使用NPUGraph（ACLGraph）时，需根据算子特性判断在重放（replay）阶段是否需要更新。若需要更新，请启用相应的update机制。具体机制说明请参阅 [NPUGraph](./01.03_npugraph_intro.ipynb)。
4. 仅算子全部为NN算子时，方可使用NPUGraph（ACLGraph）。


## 5. 课后练习


**（1）【单选题】 在 FX Graph 中，以下哪种 op 类型表示调用子模块（如 nn.Linear）？**

- A. `placeholder`
- B. `call_function`
- C. `call_module`
- D. `get_attr`

**（2）【单选题】 AOTAutograd 的核心价值是什么？**

- A. 加速前向计算，减少 Python 开销
- B. 将反向传播计算提前捕获并编译，使后端编译器能对前后向进行联合优化
- C. 将多个算子融合为单个大算子
- D. 消除 NPU 任务的启动开销


**（3）【单选题】 以下关于 torch.compile 三种后端的描述，哪项是正确的？**

- A. Inductor 后端不支持动态形状
- B. NPUGraphs 后端支持动态形状且无额外开销
- C. NPUGraph_EX 后端基于 ACLGraph 调度和 FX 图优化，适合大模型推理
- D. 三种后端都支持优化器入图


**（4）【单选题】 torch.compile 中，由哪个组件负责将用户代码捕获为 FX Graph？**

- A. Inductor
- B. Dynamo
- C. AOTAutograd
- D. NPUGraph_EX


**运行以下代码单元查看参考答案与解析。**

In [ ]:
import os
answer_path = "answer/01.02_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 ./answer/ 目录结构。")